# HealthConnect Week 8 verification
Executed on 29 September 2026. Source calculations support the final report. Dashboard and final integration tests remain pending. Run from the extracted evidence folder.

In [1]:
from pathlib import Path
import pandas as pd
import hashlib
DATA = Path('data/HealthConnect_Appointment_Data.csv')
if not DATA.exists():
    DATA = Path('evidence/data/HealthConnect_Appointment_Data.csv')
x = pd.read_csv(DATA, keep_default_na=False, na_values=[''])
e = x[x.appointment_outcome.isin(['Attended','No-Show'])].copy()
e['no_show'] = e.appointment_outcome.eq('No-Show').astype(int)
print('Source SHA-256:', hashlib.sha256(DATA.read_bytes()).hexdigest())
print('Rows:', len(x), 'Eligible:', len(e))
print(x.appointment_outcome.value_counts().to_string())
print('No-show rate:', e.no_show.mean())

Source SHA-256: 1f2c40ae14833b46fe6e1a353446a0e038f19ae06434b37d98545b3e28697137
Rows: 5000 Eligible: 4737
appointment_outcome
No-Show      2423
Attended     2314
Cancelled     263
No-show rate: 0.5115051720498206


## Segment results
Lead time bands align with the shared Data Science definitions. Group rates are descriptive, not individual calibrated probabilities.

In [2]:
e['lead_time_band'] = pd.cut(e.booking_lead_days, [-1,7,21,40,60], labels=['0-7','8-21','22-40','41-60'])
e['high_history'] = e.previous_no_shows.ge(2)
for keys in ['lead_time_band',['lead_time_band','high_history'],'reminder_sent']:
    print(e.groupby(keys, observed=True).no_show.agg(['size','sum','mean']).to_string())
    print()

                size   sum      mean
lead_time_band                      
0-7              604   178  0.294702
8-21            1148   448  0.390244
22-40           1484   769  0.518194
41-60           1501  1028  0.684877

                             size  sum      mean
lead_time_band high_history                     
0-7            False          532  144  0.270677
               True            72   34  0.472222
8-21           False         1010  374  0.370297
               True           138   74  0.536232
22-40          False         1340  683  0.509701
               True           144   86  0.597222
41-60          False         1345  898  0.667658
               True           156  130  0.833333

               size   sum      mean
reminder_sent                      
No             1285   702  0.546304
Yes            3452  1721  0.498552



## Source consistency
The selected source has 86 eligible missing distances. This does not close the discrepancy with the reported working-notebook count of 39.

In [3]:
print('Duplicate appointment IDs:', x.appointment_id.duplicated().sum())
print('Invalid history:', (x.previous_no_shows > x.previous_appointments).sum())
a = pd.to_datetime(x.appointment_date)
b = pd.to_datetime(x.booking_date)
print('Lead-time discrepancies:', ((a-b).dt.days != x.booking_lead_days).sum())
print('Missing eligible distances:', e.distance_to_clinic_km.isna().sum())
print('Median known eligible distance:', e.distance_to_clinic_km.median())
assert len(e)==4737 and e.no_show.sum()==2423

Duplicate appointment IDs: 0
Invalid history: 0
Lead-time discrepancies: 0
Missing eligible distances: 86
Median known eligible distance: 8.7


## Reproduce the full evidence tables
This script creates all summary CSVs, figures and check outputs. It keeps the original source unchanged.

In [4]:
import runpy
script = Path('reproduce_analysis.py')
if not script.exists(): script = Path('evidence/reproduce_analysis.py')
runpy.run_path(str(script), run_name='__main__')
print('Evidence regeneration complete.')

{"total": 5000, "eligible": 4737, "no_shows": 2423, "rate": 0.5115051720498206, "sha256": "1f2c40ae14833b46fe6e1a353446a0e038f19ae06434b37d98545b3e28697137"}
        variable    n  chi_square  df      p_value  cramers_v  minimum_expected_count
  lead_time_band 4737  362.006585   3 3.747675e-78   0.276444              295.050876
   history_group 4737   70.311813   2 5.394904e-16   0.121832              249.132362
   reminder_sent 4737    8.545599   1 3.463585e-03   0.042474              627.715854
appointment_type 4737    8.422134   3 3.804741e-02   0.042166              276.976567
distance_group  appointments  no_shows     rate
      <=8.7 km          2353      1149 0.488313
       >8.7 km          2298      1227 0.533943
       Unknown            86        47 0.546512
Evidence regeneration complete.


## Model arithmetic from reported counts
Source: Week 7 Analytics Testing Report. These are reported confusion-matrix counts, not independently inspected predictions. ROC-AUC and calibration are not validated here.

In [5]:
tn, fp, fn, tp = 327, 156, 191, 292
print('Evaluation rows:',tn+fp+fn+tp)
print('Accuracy:',(tp+tn)/(tn+fp+fn+tp))
print('Precision:',tp/(tp+fp))
print('Recall:',tp/(tp+fn))
print('F1:',2*tp/(2*tp+fp+fn))
print('Illustrative 2:1 cost:',fn*2+fp)
print('Illustrative 5:1 cost:',fn*5+fp)

Evaluation rows: 966
Accuracy: 0.6407867494824017
Precision: 0.6517857142857143
Recall: 0.6045548654244306
F1: 0.6272824919441461
Illustrative 2:1 cost: 538
Illustrative 5:1 cost: 1111


## Remaining evidence
Complete dashboard_acceptance.csv and integration_log.csv with real observations. Preserve unresolved status for source-version reconciliation, stable prediction IDs and final team walkthrough until supporting evidence exists.